In [1]:
import sys
import os

zero123_dir = os.path.join("/home/vaibhav", 'diffusion_augmentation', 'zero123')
color_controlnet_dir = os.path.join("/home/vaibhav", 'diffusion_augmentation', 'color_controlnet')
#controlnet_dir = os.path.join("/home/vaibhav", 'diffusion_augmentation', 'controlnet')
device = "cuda"
color_control_device = device
zero123_device = device
control_net_device = device

sys.path.append(color_controlnet_dir)
sys.path.append(zero123_dir)
#sys.path.append(controlnet_dir)

from color_controlnet.diffusers import ControlNetModel, LineartDetector, StableDiffusionImg2ImgControlNetPalettePipeline
from color_controlnet.diffusers import UniPCMultistepScheduler
from color_controlnet.infer_palette import get_cond_color, show_anns, image_grid, HWC3, resize_in_buckets, SAMImageAnnotator
from color_controlnet.infer_palette_img2img import control_color_augment

# # Zero123 imports
from zero123.nerf import load_model_from_config, generate_angles
from zero123.ldm.util import create_carvekit_interface
from zero123.ldm.models.diffusion.ddim import DDIMSampler as Zero123DDIMSampler
from omegaconf import OmegaConf

# ControlNet imports
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import cv2
import torch
from controlnet_aux import MidasDetector, CannyDetector, SamDetector

#TODO: figure out how to add all the sys paths without control net imports failing when zero123 imports are not commented out


/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(


In [7]:
def initialize_color_controlnet_models():
    """
    Initialize all models and detectors used in the pipeline
    Returns:
        control_net: dict, containing all ControlNet models and detectors
        llava: dict, containing the LLAVA model and processor
        zero123: dict, containing the Zero123 model and Carvekit interf`ace
        color_control: dict, containing the Color Control model and SAM annotator

    """

    # Color Control model
    print('Loading Color Control model...')
    color_control = {}

    controlnet = ControlNetModel.from_config("./model_configs/controlnet_config.json").half()
    adapter = ControlNetModel.from_config("./model_configs/controlnet_config.json").half()

    sketch_method = "skmodel"
    sam_annotator = SAMImageAnnotator()

    model_ckpt = f"./models/color_img2img_palette.pt"
    model_sd = torch.load(model_ckpt, map_location="cpu")["module"]

    # assign the weights of the controlnet and adapter separately
    controlnet_sd = {}
    adapter_sd = {}
    for k in model_sd.keys():
        if k.startswith("controlnet"):
            controlnet_sd[k.replace("controlnet.", "")] = model_sd[k]
        if k.startswith("adapter"):
            adapter_sd[k.replace("adapter.", "")] = model_sd[k]

    msg_control = controlnet.load_state_dict(controlnet_sd, strict=True)
    print(f"msg_control: {msg_control} ")
    if adapter is not None:
        msg_adapter = adapter.load_state_dict(adapter_sd, strict=False)
        print(f"msg_adapter: {msg_adapter} ")

    # define the inference pipline
    # sdv15_path = "/home/pat/diffusion_augmentation/color_controlnet/model_configs/sd15_config.json"
    pipe = StableDiffusionImg2ImgControlNetPalettePipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        adapter=adapter,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(color_control_device)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

    color_control['pipe'] = pipe
    color_control['sam_annotator'] = sam_annotator
    color_control['adapter'] = adapter 

    return color_control

In [8]:
color_control_model = initialize_color_controlnet_models()
color_control_model

Loading Color Control model...


/tmp/ipykernel_94573/4244897687.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_sd = torch.load(model_ckpt, map_location="cpu")["module"]


msg_control: <All keys matched successfully> 
msg_adapter: <All keys matched successfully> 


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 51909.70it/s]
You have disabled the safety checker for <class 'color_controlnet.diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img_controlnet_palette.StableDiffusionImg2ImgControlNetPalettePipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
The config attributes {'skip_prk_steps': True, 'set_alpha_to_one': False, 'steps_offset': 1, 'clip_sample': False} were passed to UniPCMultistepScheduler, but are not expected and will be ignored. Please verify

{'pipe': StableDiffusionImg2ImgControlNetPalettePipeline {
   "_class_name": "StableDiffusionImg2ImgControlNetPalettePipeline",
   "_diffusers_version": "0.15.0.dev0",
   "adapter": [
     "models",
     "ControlNetModel"
   ],
   "controlnet": [
     "models",
     "ControlNetModel"
   ],
   "feature_extractor": [
     "transformers",
     "CLIPImageProcessor"
   ],
   "requires_safety_checker": true,
   "safety_checker": [
     null,
     null
   ],
   "scheduler": [
     "diffusers",
     "PNDMScheduler"
   ],
   "text_encoder": [
     "transformers",
     "CLIPTextModel"
   ],
   "tokenizer": [
     "transformers",
     "CLIPTokenizer"
   ],
   "unet": [
     "diffusers",
     "UNet2DConditionModel"
   ],
   "vae": [
     "diffusers",
     "AutoencoderKL"
   ]
 },
 'sam_annotator': <color_controlnet.infer_palette.SAMImageAnnotator at 0x70f9baf04a30>,
 'adapter': ControlNetModel(
   (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (time_proj): Timestep

In [9]:
from IPython.display import display
import cv2

classes = ["001.ak47", "002.american-flag", "003.backpack", "004.baseball-bat", "005.baseball-glove"]

for class_name in classes:
    input_directory = os.path.join("torch/caltech256/256_ObjectCategories", class_name)
    output_directory = os.path.join("augmented_images/caltech256/color_controlnet", class_name)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    count = 0
    for file in os.listdir(input_directory):
        if count >= 2:
            break
        
        img = Image.open(os.path.join(input_directory, file))

        caption = class_name.split(".")[-1]
        color_augmented = control_color_augment(img, color_control_model['adapter'], color_control_model['pipe'], caption, color_control_model['sam_annotator'], 1, color_control_device)

        augmented_image = color_augmented[0]
        output_path = os.path.join(output_directory, file)

        augmented_image.save(output_path)
        print("Writing " + output_path + "...")

        count += 1

    

100%|██████████| 22/22 [00:01<00:00, 16.41it/s]


Writing augmented_images/caltech256/color_controlnet/001.ak47/001_0072.jpg...


100%|██████████| 22/22 [00:01<00:00, 16.07it/s]


Writing augmented_images/caltech256/color_controlnet/001.ak47/001_0003.jpg...


100%|██████████| 22/22 [00:00<00:00, 23.60it/s]


...

Writing augmented_images/caltech256/color_controlnet/004.baseball-bat/004_0049.jpg...


100%|██████████| 22/22 [00:00<00:00, 23.53it/s]


Writing augmented_images/caltech256/color_controlnet/005.baseball-glove/005_0138.jpg...


100%|██████████| 22/22 [00:00<00:00, 23.44it/s]

Writing augmented_images/caltech256/color_controlnet/005.baseball-glove/005_0009.jpg...


In [11]:
def initialize_zero123_models():
    """
    Initialize all models and detectors used in the pipeline
    Returns:
        control_net: dict, containing all ControlNet models and detectors
        llava: dict, containing the LLAVA model and processor
        zero123: dict, containing the Zero123 model and Carvekit interface
        color_control: dict, containing the Color Control model and SAM annotator

    """

    # Zero123 models
    # print('Loading Zero123 models...')
    zero123 = {}
    config_path = './model_configs/sd-objaverse-finetune-c_concat-256.yaml'
    config = OmegaConf.load(config_path)

    model_path = "./models/105000.ckpt"
    model = load_model_from_config(config, model_path, zero123_device)
    model = model.to(zero123_device)

    # print('Creating Carvekit interface...')
    carvekit_interface = create_carvekit_interface()

    zero123['model'] = model
    zero123['carvekit_interface'] = carvekit_interface 
    
    return zero123


In [12]:
zero123_model = initialize_zero123_models()
zero123_model

Loading model from ./models/105000.ckpt


/home/vaibhav/diffusion_augmentation/zero123/nerf.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pl_sd = torch.load(ckpt, map_location='cpu')
/home/vaibhav/miniconda3/

Global Step: 105000
LatentDiffusion: Running in eps-prediction mode


/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/pytorch_lightning/core/lightning.py:2058: DeprecationWarning: `torch.distributed._sharded_tensor` will be deprecated, use `torch.distributed._shard.sharded_tensor` instead
  from torch.distributed._sharded_tensor import pre_load_state_dict_hook, state_dict_hook


DiffusionWrapper has 859.53 M params.
Keeping EMAs of 688.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels


/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/carvekit/ml/wrap/tracer_b7.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(model_path, map

{'model': LatentDiffusion(
   (model): DiffusionWrapper(
     (diffusion_model): UNetModel(
       (time_embed): Sequential(
         (0): Linear(in_features=320, out_features=1280, bias=True)
         (1): SiLU()
         (2): Linear(in_features=1280, out_features=1280, bias=True)
       )
       (input_blocks): ModuleList(
         (0): TimestepEmbedSequential(
           (0): Conv2d(8, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
         )
         (1-2): 2 x TimestepEmbedSequential(
           (0): ResBlock(
             (in_layers): Sequential(
               (0): GroupNorm32(32, 320, eps=1e-05, affine=True)
               (1): SiLU()
               (2): Conv2d(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
             )
             (h_upd): Identity()
             (x_upd): Identity()
             (emb_layers): Sequential(
               (0): SiLU()
               (1): Linear(in_features=1280, out_features=320, bias=True)
             )
             (ou

In [13]:
# Define the angles for augmentation
angles = [
    ("right", 0, 15, 0),
]

from IPython.display import display
import cv2

classes = ["001.ak47", "002.american-flag", "003.backpack", "004.baseball-bat", "005.baseball-glove"]

for class_name in classes:

    input_directory = os.path.join("torch/caltech256/256_ObjectCategories", class_name)
    output_directory = os.path.join("augmented_images/caltech256/zero123", class_name)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    count = 0
    for file in os.listdir(input_directory):
        if count >= 2:
            break
        
        img = Image.open(os.path.join(input_directory, file))

        # Apply Zero123 augmentations
        preprocessed_image, augmented_images = generate_angles(
            input_image=img,
            angles=angles,
            model=zero123_model['model'],  # Your loaded model
            carvekit_interface=zero123_model['carvekit_interface'],  # Your Carvekit interface
            device=zero123_device,  # Your device (e.g., 'cuda:0')
            precision='autocast',  # Use 'autocast' for mixed precision if supported
            h=256,
            w=256,
            ddim_steps=50,
            scale=3.0,
            n_samples=1,
            ddim_eta=1.0
        )
        output_path = os.path.join(output_directory, file)

        augmented_images[0].save(output_path)
        print("Writing " + output_path + "...")
        

        count += 1


old input_im: (700, 514)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 46.34it/s]


Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/001.ak47/001_0072.jpg...
old input_im: (300, 186)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 48.35it/s]


Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/001.ak47/001_0003.jpg...
old input_im: (237, 235)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


...

Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/004.baseball-bat/004_0049.jpg...
old input_im: (191, 161)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 48.25it/s]


Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/005.baseball-glove/005_0138.jpg...
old input_im: (200, 200)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 48.30it/s]

Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/005.baseball-glove/005_0009.jpg...


In [2]:
canny = CannyDetector()
midas = MidasDetector.from_pretrained("lllyasviel/Annotators")
sam = SamDetector.from_pretrained("ybelkada/segment-anything", subfolder="checkpoints")

def preprocess_image(image_path, preprocessor_type):
    """
    Processes an image with the specified preprocessor type.

    Parameters:
    - image_path (str): Path to the input image.
    - preprocessor_type (str): Type of preprocessor ('canny', 'midas', 'segmentation').

    Returns:
    - processed_image: The processed image (PIL.Image).
    """
    img = Image.open(image_path).convert("RGB").resize((512, 512))
    
    if preprocessor_type == "Canny":
        # Canny Edge Detection
        np_img = np.array(img)
        low_threshold = 100
        high_threshold = 200
        edges = cv2.Canny(np_img, low_threshold, high_threshold)
        edges = edges[:, :, None]
        edges = np.concatenate([edges, edges, edges], axis=2)
        processed_image = Image.fromarray(edges)
    elif preprocessor_type == "Midas":
        # Midas Depth Estimation
        processed_image = midas(img)
    elif preprocessor_type == "Segmentation":
        # Segmentation using SAM
        processed_image = sam(img)
    else:
        raise ValueError(f"Unsupported preprocessor type: {preprocessor_type}")
    
    return processed_image


/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/controlnet_aux/midas/midas/base_model.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  parameters = to

In [12]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
import torch
from diffusers import UniPCMultistepScheduler

classes = ["001.ak47", "002.american-flag", "003.backpack", "004.baseball-bat", "005.baseball-glove"]

controlnet_models = {
    "Canny": "lllyasviel/sd-controlnet-canny",
    "Midas": "lllyasviel/sd-controlnet-depth",
    "Segmentation": "lllyasviel/sd-controlnet-seg"
}

for class_name in classes:

    # Get all images in the class directory
    class_dir = f"torch/caltech256/256_ObjectCategories/{class_name}"

    image_files = [f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    count = 0
    for image_file in image_files:
        if count >= 2:
            break

        image_path = os.path.join(class_dir, image_file)
        
        # Process image with each preprocessor type
        preprocessor_types = ["Canny", "Midas", "Segmentation"]
        processed_images = {}

        for preprocessor_type in preprocessor_types:
            processed_image = preprocess_image(image_path, preprocessor_type)
            processed_images[preprocessor_type] = processed_image

        # Process with each ControlNet model
        for preprocessor_type, model_id in controlnet_models.items():
            # Load the specific ControlNet model
            controlnet = ControlNetModel.from_pretrained(
                model_id,
                torch_dtype=torch.float16
            ).to(control_net_device)

            pipe = StableDiffusionControlNetPipeline.from_pretrained(
                "runwayml/stable-diffusion-v1-5",
                controlnet=controlnet,
                torch_dtype=torch.float16
            ).to(control_net_device)

            pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
            pipe.enable_xformers_memory_efficient_attention()

            # Create prompt based on class name
            class_prompt = class_name.split('.')[1].replace('-', ' ')
            prompt = [f"{class_prompt}, best quality, extremely detailed"]
            negative_prompt = ["monochrome, lowres, bad anatomy, worst quality, low quality"]

            generator = [torch.Generator(device=control_net_device).manual_seed(2) for _ in range(len(prompt))]

            output = pipe(
                prompt,
                processed_images[preprocessor_type],
                negative_prompt=negative_prompt * len(prompt),
                generator=generator,
                num_inference_steps=20,
            )

            # Save generated images
            output_path = os.path.join(f"augmented_images/caltech256/{preprocessor_type}/{class_name}", image_file)
            for i, img in enumerate(output.images):
                img.save(output_path)
                print("Writing " + output_path + "...")


        count += 1


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 68834.31it/s]
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
100%|██████████| 20/20 [00:00<00:00, 29.58it/s]


Writing augmented_images/caltech256/Canny/001.ak47/001_0072.jpg...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 69750.07it/s]
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
100%|██████████| 20/20 [00:00<00:00, 29.49it/s]


Writing augmented_images/caltech256/Midas/001.ak47/001_0072.jpg...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 14331.33it/s]
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
100%|██████████| 20/20 [00:00<00:00, 29.60it/s]


...

Writing augmented_images/caltech256/Canny/005.baseball-glove/005_0009.jpg...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 75256.65it/s]
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
100%|██████████| 20/20 [00:00<00:00, 29.82it/s]


Writing augmented_images/caltech256/Midas/005.baseball-glove/005_0009.jpg...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 73670.44it/s]
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
100%|██████████| 20/20 [00:00<00:00, 29.38it/s]


Writing augmented_images/caltech256/Segmentation/005.baseball-glove/005_0009.jpg...
